In [ ]:
# start coding here
from scripts.utils.io import load_dict
import pandas as pd

samples = ['/'.join(file.split('/')[:-1]) for file in snakemake.input if "r_naught" not in file]
df = pd.DataFrame(index=samples)

for file in snakemake.input:
    sample = '/'.join(file.split('/')[:-1])

    dictionary = load_dict(file)

    if "eig_1" in dictionary.keys():
        for key in ['eig_1', 'clustering', 'density']:
            df.loc[sample, key] = dictionary[key]
    else:
        df.loc[sample,'peak'] = dictionary['peak'] 
        df.loc[sample,'time'] = dictionary['time']

df

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
delta = 0.1
bounds = np.arange(0.1,1.0,delta)
n_bounds = len(bounds)

matrix = np.zeros((n_bounds,n_bounds,3))


cmap = plt.get_cmap('Greens')
fig, axes = plt.subplots(1,3,figsize=(18,6),dpi=300)

for i, y in enumerate(bounds):
    for j, x in enumerate(bounds):
        i_idx = n_bounds-(i+1)
        idx = (x <= df['clustering']) & (df['clustering'] < (x + delta)) & (y <= df['density']) & (df['density'] < (y + delta))

        samples = df.loc[idx]
        matrix[i_idx,j,:] = [
            samples['peak'].mean(),
            samples['time'].mean(),
            samples['eig_1'].mean()
        ]

for i, title in enumerate([r"$I_{\max}$", 'Time-to-peak', r"$\lambda_1$"]):
    img = axes[i].imshow(matrix[:,:,i],cmap=cmap)
    plt.colorbar(img, fraction=0.046)
    axes[i].set_xlabel(r"Clustering $(\phi)$")
    axes[i].set_title(title)
    axes[i].set_yticks([])
    axes[i].set_xticks(np.arange(n_bounds))
    axes[i].set_xticklabels([f"{value:.2f}" for value in bounds])

axes[0].set_ylabel(r"Density $(\rho)$")
axes[0].set_yticks(np.arange(n_bounds))
axes[0].set_yticklabels([f"{value:.2f}" for value in np.flip(bounds)])
plt.tight_layout()

plt.savefig(snakemake.output[0], bbox_inches='tight')